# 11. Evaluación final bloqueada en test

**Fases del guía metodológica cubiertas: 17 (Evaluación final bloqueada en test)**

> Regla central: *definir -> auditar -> dividir -> aprender solo con train ->
> seleccionar con validación/CV -> comprobar una vez con test -> empaquetar -> monitorizar*.



## 17.1 Protocolo

```
Decisiones congeladas (fase 16)
-> cargar test intacto (data/processed/X_test.csv)
-> aplicar transformaciones ya aprendidas (pipeline guardado)
-> inferir
-> calcular métricas finales
-> documentar
```

El test solo se toca UNA vez. Si algo sale mal, se crea un test nuevo; este deja de valer.

### 17.1.1 Carga del pipeline y predicción sobre test

Cargamos el pipeline final (sin volver a entrenarlo) y los metadatos con el umbral
congelado. Aplicamos el feature engineering a `X_test` (función pura, sin estado) y
generamos las probabilidades. Este es el **único** uso del test en todo el proyecto.


In [1]:

import sys, pathlib
ROOT = pathlib.Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import joblib, json
from src.data.load_data import load_processed
from src.features.build_features import add_domain_features
from src.evaluation.metrics import compute_metrics, calibration_summary, find_optimal_threshold, roc_curve_data, pr_curve_data

d = load_processed()
Xte = add_domain_features(d["X_test"]); yte = d["y_test"]

meta = json.loads((ROOT / "models" / "final_model_metadata.json").read_text(encoding="utf-8"))
umbral = meta["threshold"]
print("Umbral congelado:", umbral)

# Mismas probabilidades calibradas que usa la API en producción
from src.api.main import predict_proba_series
y_proba = predict_proba_series(Xte)
print("Predicciones sobre test (bloqueado):", Xte.shape[0], "registros")


Umbral congelado: 0.33999999999999997


Predicciones sobre test (bloqueado): 133 registros



### 17.1.2 Métricas finales

Calculamos el bloque completo de métricas con el umbral congelado: ROC-AUC (primaria),
PR-AUC, accuracy, precision, recall, F1, Brier y el coste de decisión medio. La matriz de
confusión (TP/FP/FN/TN) se incluye en el dict. Estas cifras son las que se reportan en el
README y en la model card: son la **única** evaluación honesta del modelo.


In [2]:

# Métricas finales con el umbral de coste congelado
metrics = compute_metrics(yte, y_proba, threshold=umbral)
for k, v in metrics.items():
    print(f"{k:12s}: {v:.4f}" if isinstance(v, float) else f"{k:12s}: {v}")


roc_auc     : 0.7657
pr_auc      : 0.7411
accuracy    : 0.7444
precision   : 0.6452
recall      : 0.7692
f1          : 0.7018
brier       : 0.1868
threshold   : 0.3400
cost        : 0.3459
n_tp        : 40
n_fp        : 22
n_fn        : 12
n_tn        : 59



### 17.1.3 Curvas ROC, PR y matriz de confusión

Dibujamos las tres figuras que resumen el comportamiento en test: la **curva ROC** (con
su AUC), la **curva Precision-Recall** (relevante por el desbalance y la priorización de
intervenciones) y la **matriz de confusión** con el umbral de coste. La figura combinada
se guarda en `reports/figures/11_test_evaluacion.png`.


In [3]:

# Curvas ROC / PR y matriz de confusión (fase 17.2)
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))

roc = roc_curve_data(yte, y_proba)
axes[0].plot(roc["fpr"], roc["tpr"], label=f"ROC-AUC = {metrics['roc_auc']:.3f}")
axes[0].plot([0, 1], [0, 1], "--", color="gray"); axes[0].set_xlabel("FPR"); axes[0].set_ylabel("TPR")
axes[0].set_title("Curva ROC (test)"); axes[0].legend()

pr = pr_curve_data(yte, y_proba)
axes[1].plot(pr["recall"], pr["precision"], label=f"PR-AUC = {metrics['pr_auc']:.3f}")
axes[1].set_xlabel("Recall"); axes[1].set_ylabel("Precision"); axes[1].set_title("Curva PR (test)"); axes[1].legend()

cm = confusion_matrix(yte, (y_proba >= umbral).astype(int))
ConfusionMatrixDisplay(cm, display_labels=["bajo", "alto"]).plot(ax=axes[2], cmap="Blues")
axes[2].set_title(f"Matriz de confusión (umbral={umbral:.2f})")

plt.tight_layout(); plt.savefig(ROOT / "reports" / "figures" / "11_test_evaluacion.png", dpi=120)
plt.show()


C:\Users\sgml1\AppData\Local\Temp\ipykernel_15020\2839523479.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()



### 17.1.4 Calibración

La **calibración** mide si la probabilidad predicha coincide con la frecuencia real del
evento. Calculamos el Brier score y el ECE (Expected Calibration Error) sobre test. Un
ECE < 0.10 se considera bueno; el valor obtenido (~0.12) queda documentado como mejora
futura (recalibración isotónica), sin alterar el modelo ya evaluado.


In [4]:

# Calibración
cal = calibration_summary(yte, y_proba)
print("Brier (test):", round(cal["brier"], 4), "| ECE:", round(cal["ece"], 4))


Brier (test): 0.1868 | ECE: 0.1292



### 17.1.5 Matriz de confusión multiclase (target original 1-5)

Hasta aquí la evaluación se ha hecho sobre el **target binario** (consumo alto vs bajo),
que es la definición operativa del proyecto. Sin embargo, el **target original `Walc` es
ordinal multiclase (5 niveles)**: 1 = muy bajo, 5 = muy alto. Para mostrar el rendimiento
por nivel de consumo, entrenamos un clasificador multiclase equivalente (RandomForest
sobre los 5 niveles) con el mismo pipeline y partición, y dibujamos su **matriz de
confusión 5x5**. Esta matriz complementa a la binaria: muestra qué niveles se confunden
entre sí (por ejemplo, si el modelo confunde 3 con 2 o con 4, que son los límites de la
binarización).

Nota: el rendimiento multiclase (accuracy ~0.47) es inferior al binario porque predecir
el nivel exacto es más difícil que predecir alto/bajo; es una vista adicional, no el
modelo de producción.


In [5]:

# Matriz de confusión MULTICLASE (5 niveles del target original)
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score
from src.models.train_model import make_pipeline
from src.evaluation.metrics import multiclass_evaluation

# Reconstruir el target ordinal 1-5 con la MISMA partición (semilla 42)
from src.data.load_data import load_student_dataset
from src.features.build_features import build_dataset_unique_students
from src.data.make_dataset import stratified_split

mat_raw, por_raw, _ = load_student_dataset()
mc = ["school","sex","age","address","famsize","Pstatus","Medu","Fedu","Mjob","Fjob","reason","nursery","internet"]
df_ord = build_dataset_unique_students(mat_raw, por_raw, mc)
X_ord = add_domain_features(df_ord.drop(columns=["Walc"]))
y_ord = df_ord["Walc"].astype(int)  # target ordinal 1-5
split_ord = stratified_split(X_ord, y_ord, 0.20, 0.20, random_state=42)

pipe_multi = make_pipeline(RandomForestClassifier(
    n_estimators=300, max_depth=10, min_samples_leaf=5, random_state=42))
pipe_multi.fit(pd.concat([split_ord["X_train"], split_ord["X_val"]]),
               pd.concat([split_ord["y_train"], split_ord["y_val"]]))
y_pred_ord = pipe_multi.predict(split_ord["X_test"])

# Evaluación multiclase completa: accuracy exacta, +/-1 nivel, error ordinal
res_multi = multiclass_evaluation(split_ord["y_test"], y_pred_ord, labels=[1, 2, 3, 4, 5])
print("Accuracy multiclase exacta (5 niveles):", round(res_multi["accuracy"], 4))
print("Accuracy dentro de +/-1 nivel         :", round(res_multi["accuracy_1off"], 4))
print("Error ordinal medio                   :", round(res_multi["error_medio"], 4))
print("Errores por distancia                 :", res_multi["errores_por_distancia"])
print("(Confundir niveles vecinos es lo habitual; confundir 1 con 5 es rarísimo ->",
      "la granularidad de 5 clases aporta poco frente a agrupar.)")

# Matriz 5x5
fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay.from_predictions(
    split_ord["y_test"], y_pred_ord, labels=[1, 2, 3, 4, 5],
    cmap="Blues", ax=ax, colorbar=False,
)
ax.set_title("Matriz de confusión MULTICLASE (Walc 1-5, test)")
ax.set_xlabel("Predicho"); ax.set_ylabel("Real")
plt.tight_layout()
plt.savefig(ROOT / "reports" / "figures" / "11_matriz_multiclase.png", dpi=120)
plt.show()


Accuracy multiclase exacta (5 niveles): 0.4887
Accuracy dentro de +/-1 nivel         : 0.8271
Error ordinal medio                   : 0.7368
Errores por distancia                 : {0: 65, 1: 45, 2: 16, 3: 7}
(Confundir niveles vecinos es lo habitual; confundir 1 con 5 es rarísimo -> la granularidad de 5 clases aporta poco frente a agrupar.)


C:\Users\sgml1\AppData\Local\Temp\ipykernel_15020\63972041.py:45: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()



### 17.1.6 Guardado del informe

Persistimos `reports/final_evaluation.json` (métricas + resumen de errores) y
`reports/test_predictions.csv` (predicciones trazables). Estos ficheros son la fuente de
verdad para README, model card e informe técnico.


In [6]:

# Guardar informe de evaluación final
import pandas as pd
from src.evaluation.metrics import save_evaluation_report, error_analysis
err = error_analysis(yte, y_proba, Xte, threshold=umbral, top_k=5)
save_evaluation_report(metrics, pd.DataFrame(), {"n_fp": err["n_fp"], "n_fn": err["n_fn"]},
                       ROOT / "reports" / "final_evaluation.json")
print("Informe guardado en reports/final_evaluation.json")
print("Errores: FP =", err["n_fp"], "| FN =", err["n_fn"])


Informe guardado en reports/final_evaluation.json
Errores: FP = 22 | FN = 12



## 17.2 Presentación: tabla resumen baseline vs. candidatos vs. final

| Modelo | ROC-AUC CV | ROC-AUC test |
|---|---|---|
| Dummy (referencia) | 0.50 | — |
| Regla de negocio | — | (ver notebook 06) |
| Logistic Regression | (fase 06) | — |
| RandomForest final | (fase 09) | **ver celdas anteriores** |

## 17.3 Regla

Si se cambia cualquier decisión por el resultado del test, el test pasa a ser validation
y se debe crear un **nuevo test bloqueado**. Este proyecto no modifica ninguna decisión.
